In [ ]:
# ar/data-analysis/normal/04-filtering-rows
# Generated companion notebook for the PyDA course.
# Run cells top-to-bottom (or in any order) to follow the lesson.

print("PyDA — ready 🚀")


In [ ]:
# 💾 Load the course datasets into this environment
# The course data files live in the PyDA repo; pull them so
# `open("…")` / `pd.read_csv("…")` work exactly like on disk.
import os
def _fetch(name, aliases=()):
    if os.path.exists(name):
        return
    url = f"https://raw.githubusercontent.com/abderrahim-lectures/python-data-analysis-course/main/public/datasets/{name}"
    os.system(f"curl -sL -o {name} {url}")
    for alias in aliases:
        if not os.path.exists(alias):
            os.system(f"cp {name} {alias}")

_fetch("titanic.csv", ())


## التصفية بالشروط المنطقية

التصفية هي طريقة تركيزك على الجزء المهم من البيانات. تنشئ **قناعًا منطقيًا** (boolean mask) — كائن Series من قيم True/False — وتستخدمه لتحديد الصفوف.


In [ ]:
import pandas as pd

# titanic.csv ships with the course — load it from the browser file system.
df = pd.read_csv("titanic.csv")

# Filter passengers older than 30
older = df[df["Age"] > 30]
print(older.shape)   # fewer rows than original 891


التعبير `df["Age"] > 30` يُنتج كائن Series منطقيًا:


In [ ]:
0       True
1       True
2      False
3       True
...


تمريره داخل `df[...]` يحتفظ فقط بالصفوف التي تكون قيمتها `True`.

## دمج الشروط

استخدم `&` (و) و `|` (أو) لدمج الشروط. **يجب وضع كل شرط بين قوسين:**


In [ ]:
# Female passengers in first class
first_class_female = df[(df["Sex"] == "female") & (df["Pclass"] == 1)]
print(first_class_female.head())


In [ ]:
# Passengers younger than 25 OR older than 60
young_or_old = df[(df["Age"] < 25) | (df["Age"] > 60)]
print(young_or_old.shape)


خطأ شائع: استخدام `and`/`or` بدلًا من `&`/`|`. عاملَا بايثون `and`/`or` لا يعملان على مستوى العناصر على كائنات Series في pandas وسيسببان خطأ.

## استخدام .isin() لقيم متعددة

عندما تحتاج إلى المطابقة مع قائمة قيم، استخدم `.isin()`:


In [ ]:
# Passengers who embarked from Cherbourg or Southampton
embarked_filter = df[df["Embarked"].isin(["C", "S"])]


In [ ]:
# Passengers in class 1 or 2
upper_classes = df[df["Pclass"].isin([1, 2])]


## استخدام .between() للنطاقات

طريقة `.between()` أنظف من ربط مقارنتين:


In [ ]:
# Passengers aged 20 to 30 (inclusive by default)
twenties = df[df["Age"].between(20, 30)]
print(twenties.shape)


هذا مكافئ لـ `df[(df["Age"] >= 20) & (df["Age"] <= 30)]` لكنه أكثر قابلية للقراءة.

## التصفية بأساليب النصوص

تتيح لك أداة `.str` تطبيق عمليات النصوص على عمود كامل:


In [ ]:
# Passengers whose name contains "Master" (a title)
masters = df[df["Name"].str.contains("Master", na=False)]
print(masters.shape)


In [ ]:
# Passengers whose ticket starts with "A"
a_tickets = df[df["Ticket"].str.startswith("A", na=False)]


الوسيط `na=False` يعالج القيم المفقودة بلطف — بدونها، ستسبب قيم NaN أخطاء.

## التصفية باستخدام .query()

بالنسبة للفلاتر المعقدة، يوفر `.query()` بديلًا مقروءًا:


In [ ]:
# Equivalent to df[(df["Age"] > 25) & (df["Survived"] == 1)]
survivors_over_25 = df.query("Age > 25 and Survived == 1")


قراءة هذا سطرية تقريبًا مثل نص إنجليزي وتتجنب تكرار صياغة `df["column"]`.

## تخزين الفلاتر في متغيرات

بالنسبة للشروط المعقدة، خزّن القناع المنطقي في متغير أولًا:


In [ ]:
is_female = df["Sex"] == "female"
is_first_class = df["Pclass"] == 1
is_survived = df["Survived"] == 1

# Combine them
result = df[is_female & is_first_class & is_survived]
print(f"Female first-class survivors: {len(result)}")


هذا الأسلوب يجعل شفرتك أسهل بكثير في القراءة وتصحيح الأخطاء.

## جرّب بنفسك

من مجموعة بيانات تيتانيك، صفِّ للعثور على:
1. جميع الركاب الذين دفعوا ثمن تذكرة أكثر من 100
2. جميع الركابات في الدرجة الثالثة
3. جميع الركاب الذين يحوي اسمهم لقب "Mrs"


In [ ]:
import pandas as pd

# titanic.csv ships with the course — load it from the browser file system.
df = pd.read_csv("titanic.csv")

high_fare = df[df["Fare"] > 100]
print(f"High fare passengers: {len(high_fare)}")

third_class_female = df[(df["Sex"] == "female") & (df["Pclass"] == 3)]
print(f"Third-class females: {len(third_class_female)}")

mrs = df[df["Name"].str.contains("Mrs", na=False)]
print(f"Passengers with title Mrs: {len(mrs)}")


## خلاصات رئيسية

- الفهرسة المنطقية `df[mask]` هي آلية التصفية الأساسية في pandas
- استخدم `&` للـAND و `|` للـOR — لفّ كل شرط على حدة بين قوسين دائمًا
- `.isin()` يطابق قائمة؛ `.between()` يتعامل مع النطاقات بنظافة
- `.str.contains()` يصفّي بمطابقة نص فرعي — استخدم `na=False` للأمان

## تحدي التطبيق

من مجموعة بيانات تيتانيك، ابحث عن جميع الركاب الذين: (1) كانوا ذكورًا، (2) في الدرجة الثانية أو الثالثة، (3) تتراوح أعمارهم بين 18 و 35 عامًا، و(4) نجوا. كم عدد الركاب المطابقين للشروط الأربعة جميعها؟


In [ ]:
# The end. Practice on your own — each cell is a minimal, runnable chunk.
